## 🌻 **Denop Deep Dive**

This is a more advanced version of the [Density Optimization notebook](tutorial_6_density_optimization.ipynb). It can also be read after the tutorial notebooks in `notebooks/tutorial/` in order to gain a deeper understanding of what is happening under the hood.

Instead of making use of the `density_optimization` method, the `SampleGenerator`, `FunctionalFactory`, `TorchOptimizer` and `MLDFTLitModule`, we build those objects by hand with fixed arguments. With a debugger, you find that arguments are shuffled around through many layers of abstraction and indirections. Here, we roll up the main pieces into a more serialized code for ease of understanding.

### 🎈 **Install & import dependencies**

In [ ]:
# %pip install -e ../../
# or you may want to use the great `uv` (https://docs.astral.sh/uv/guides/integration/jupyter/):
# !uv sync

In [ ]:
import torch
from omegaconf import DictConfig, OmegaConf
from pyscf import gto

from mldft.utils.utils import set_default_torch_dtype

### 🎈 **Download our latest models from [Hugging Face](https://huggingface.co/sciai-lab/structures25/tree/main)**

In [ ]:
# https://huggingface.co/docs/datasets/cache#cache-directory
# The default cache directory is `~/.cache/huggingface/datasets`
# You can change it by setting this variable to any path you like
CACHE_DIR = None

Here, we only work with the QM9 dataset.

In [ ]:
from huggingface_hub import hf_hub_download

# https://huggingface.co/sciai-lab/structures25/tree/main
REPO_ID = "sciai-lab/structures25"


MODEL_CONFIG_PATH = hf_hub_download(
    repo_id=REPO_ID, filename="trained-on-qm9/hparams.yaml", cache_dir=CACHE_DIR
)
MODEL_PATH = hf_hub_download(
    repo_id=REPO_ID,
    filename="trained-on-qm9/trained-on-qm9.ckpt",
    cache_dir=CACHE_DIR,
)
MOLECULE_FILE = hf_hub_download(
    repo_id=REPO_ID, filename="molecule-samples/molecule.xyz", cache_dir=CACHE_DIR
)

### 🎈 **Denop building blocks**

In the codebase, we use a custom class called `TorchOptimizer` that wraps around PyTorch's native optimizer, e.g. the [SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html) optimizer. This optimizer is passed to the `density_optimization` function, such that the latter can call the optimizer's `optimize` method. Note that in this notebook we instead define a custom `DenopOptimizer`, which  doesn't take any configurations, but hardcodes all optimization parameters like learning rate, momentum etc.

The `optimize` method essentially does the gradient descent step, i.e. it updates the density coefficients based on the gradient that is computed by the `energy_functional`. If the norm of the gradient falls under some threshold, the optimization stops.

The `energy_functional` is passed to the `optimize` method as `Callable`. We construct this functional later in this notebook.

In [ ]:
from collections.abc import Callable

from mldft.ml.data.components.of_data import OFData
from mldft.ml.models.components.loss_function import project_gradient
from mldft.ofdft.energies import Energies


class DenopOptimizer:
    """
    The optimizer in analogy to our `TorchOptimizer`.
    """

    @classmethod
    def optimize(
        cls,
        sample: OFData,
        energy_functional: Callable,
    ) -> tuple[Energies, bool]:
        parameters = sample.coeffs.clone()
        torch_optimizer = torch.optim.SGD([parameters], lr=3e-3, momentum=0.9)  # type: ignore

        converged = False
        for step in range(10_000):
            energy, gradient = energy_functional(sample)
            gradient_projected = project_gradient(gradient, sample)
            gradient_norm = torch.norm(gradient_projected).item()
            print(f"Step {step} | Gradient norm: {gradient_norm:.6f}", end="\r")

            if gradient_norm < 0.0001:
                converged = True
                break

            # bring sample.coeffs closer to parameters via gradient descent
            torch_optimizer.zero_grad()
            parameters.grad = gradient_projected
            torch_optimizer.step()
            update = parameters - sample.coeffs
            sample.coeffs += project_gradient(update, sample)

        return energy, converged

In the codebase, the `MLDFTLitModule` is — as far as density optimization concerns — only used as a wrapper for the trained neural network. The latter, in all its complexity, is stored in a checkpoint (`.ckpt`) file. Since the `MLDFTLitModule` inherits from `LightningModule`, we can call `MLDFTLitModule.load_from_checkpoint(path_to_checkpoint_file)` later on in order to load it from such a `ckpt` file. The network will become available as `net` parameter of the `MLDFTLitModule`. We use this `net` to call `self.net(sample)`, which does the forward pass of the network. Via the awesome PyTorch [autograd](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html), we numerically differentiate on the fly and obtain our desired gradients necessary for gradient descent.

In [ ]:
from lightning import LightningModule

from mldft.ml.data.components.of_data import OFData, Representation


class MLDFTLitModuleSimplified(LightningModule):
    """
    A simplified version of the MLDFTLitModule that only does a forward pass
    on a given sample.
    """

    def __init__(self, net: torch.nn.Module) -> None:
        super().__init__()
        self.net = net

    def forward(self, sample: OFData) -> OFData:
        with torch.enable_grad():
            sample.coeffs.requires_grad = True
            pred_energy, pred_diff = self.net(sample)
            pred_gradients = torch.autograd.grad(
                pred_energy.sum(), sample.coeffs, create_graph=False
            )[0]

        # Just for convenience, we make predictions available directly
        # on the sample as attributes. We could just as well return them
        # from this function.
        sample.add_item("pred_energy", pred_energy, Representation.SCALAR)
        sample.add_item("pred_gradient", pred_gradients, Representation.GRADIENT)
        sample.add_item("pred_diff", pred_diff, Representation.VECTOR)

        return sample

### 🎈 **Denop eval**

Finally, let us construct the **energy functional** that is used in the optimizer: `energy, gradient = energy_functional(sample)`. This functional is essential to the density optimization procedure. It takes a `sample` (`OFData` object) as input and returns the energy and its gradient with respect to the density coefficients.

In this example, we arbitrarily consider the following contributions: `[model, "hartree", "nuclear_attraction"]`.
- For the model contribution, we use the `MLDFTLitModule` (see above) to predict the energy and gradient via the `forward` pass of the neural network. By _energy_, we mean a learned approximation of the sum of kinetic and e**xc**hange-correlation energy ($E_{\text{kin}} + E_{\text{xc}}$, where we often use $T_S$ for the kinetic term).
- For other contributions, we employ analytical expressions, e.g. for "hartree" and "nuclear_attraction". Feel free to explore more possible contributions in the [`FunctionalFactory`](../../mldft/ofdft/functional_factory.py) class.

Based on this, the `FunctionalFactory` can be seen as a wrapper around the different contributions to the energy functional. It takes care of calling the right functions for each contribution and summing them up to get the total energy and gradient.

In [ ]:
from functools import partial

from mldft.ml.data.components.basis_transforms import transform_tensor_with_sample
from mldft.ml.data.components.of_data import OFData
from mldft.ofdft.energies import Energies
from mldft.ofdft.initial_guess import initial_guess


def eval_model(model: torch.nn.Module, sample: OFData) -> tuple[float, torch.Tensor]:
    assert model.training is False
    assert model.dtype == sample.coeffs.dtype

    # This is the only time we do a forward pass on the model during denop
    sample = model.forward(sample)
    energy = sample.pred_energy.item()
    grad = sample.pred_gradient.to(torch.float64)

    # .detach() such that grad is not part of the autograd computation graph
    sample.coeffs = sample.coeffs.detach()

    return energy, grad


def eval_hartree(coeffs: torch.Tensor, coulomb_matrix: torch.Tensor) -> tuple[float, torch.Tensor]:
    hartree_potential_vector = coulomb_matrix @ coeffs
    hartree_energy = 0.5 * coeffs @ hartree_potential_vector

    return hartree_energy.item(), hartree_potential_vector


def eval_nuclear_attraction(
    coeffs: torch.Tensor, nuclear_attraction_vector: torch.Tensor
) -> tuple[float, torch.Tensor]:
    nuclear_attraction_energy = coeffs @ nuclear_attraction_vector
    return nuclear_attraction_energy.item(), nuclear_attraction_vector


def eval_energy_functional(
    sample: OFData,
    model: torch.nn.Module,
    mole: gto.Mole,
) -> tuple[Energies, torch.Tensor]:
    """
    Evaluate the energy functional for the given coefficients.

    Here, we replay what `FunctionalFactory::evaluate_functional` does under the hood
    for the fixed, arbitrary contributions: [model, "hartree", "nuclear_attraction"].
    """

    energies = Energies(mole)

    # 🎈 Example Contributions: [model, "hartree", "nuclear_attraction"]
    # In the "real" code, this is done in a loop over self.contributions
    # e.g. search for `for contribution in self.contributions`
    # inside the `FunctionalFactory` class.
    energy_model, gradient_model = eval_model(model, sample)
    energy_hartree, gradient_hartree = eval_hartree(sample.coeffs, sample.coulomb_matrix)
    energy_nuclear, gradient_nuclear = eval_nuclear_attraction(
        sample.coeffs, sample.nuclear_attraction_vector
    )

    # just to nicely report the individual results in one convenient `energy` object
    energies["kin_plus_xc"] = energy_model
    energies["hartree"] = energy_hartree
    energies["nuclear_attraction"] = energy_nuclear

    gradient = gradient_model + gradient_hartree + gradient_nuclear

    assert not sample.coeffs.requires_grad, "Coeffs should not require grad"
    assert not gradient.requires_grad, "Gradient should not require grad"

    return energies, gradient

Last, but not least, everything comes together in the `_denop` function, in analogy to the function `density_optimization` in `density_optimization.py`.

In [ ]:
def _denop(
    model: torch.nn.Module,
    sample: OFData,
    mole: gto.Mole,
) -> tuple[Energies, torch.Tensor, bool]:
    """
    Performs the density optimization procedure.

    - Initializes the coefficients with a standard initial guess (MINAO).
      In the "real" code, there are more options available.
    - Defines the energy functional to be optimized.
    - Uses the `DenopOptimizer` to minimize the energy functional.
    - Returns the final energies, coefficients, and convergence status.
    """

    sample.coeffs = initial_guess(sample, mole, "minao")
    energy_functional = partial(eval_energy_functional, model=model, mole=mole)
    energies, converged = DenopOptimizer.optimize(sample, energy_functional)
    coeffs = transform_tensor_with_sample(
        sample, sample.coeffs, Representation.VECTOR, invert=True
    )
    return energies, coeffs, converged

### 🎈 **Run Denop (interface)**

Before we can run `denop` on a molecule, we need one more step: mapping a molecule to an `OFData` object, which we refer to as `sample`. This object holds important molecule-related data like the basis set, the coefficients, atomic numbers etc. In the codebase, this is done by the `SampleGenerator` class. Here, we just hardcode some of its functionality in order to keep things simple. It is _not_ crucial for this notebook to understand how it works in detail.

In [ ]:
import numpy as np
from hydra.utils import instantiate

from mldft.ml.data.components.basis_info import calculate_basis_size
from mldft.ml.data.components.convert_transforms import (
    PrepareForDensityOptimization,
    to_torch,
)
from mldft.utils.molecules import check_atom_types


def get_sample_from_mol(mol: gto.Mole, model_config: DictConfig) -> OFData:
    """
    Returns a sample from a molecule with the appropriate transforms applied.

    Basically copied from `SampleGenerator::get_sample_from_mol`.
    """
    basis_info = instantiate(model_config.data.basis_info)
    check_atom_types(mol, basis_info.atomic_numbers)  # type: ignore
    n_basis = calculate_basis_size(mol, basis_info)

    sample = OFData.construct_new(
        basis_info=basis_info,
        pos=mol.atom_coords(unit="Bohr"),
        atomic_numbers=mol.atom_charges(),
        coeffs=np.zeros(n_basis),
        basis_integrals="infer_from_basis",
        add_irreps=True,
    )

    transforms = instantiate(model_config.data.transforms)
    transforms.pre_transforms.insert(0, PrepareForDensityOptimization(basis_info, add_grid=False))
    transforms.add_transformation_matrix = True
    transforms.use_cached_data = False

    sample = transforms.forward(sample)
    sample.mol.charge = mol.charge
    sample = to_torch(sample, device="cpu")
    return sample

Finally, let's define a nice interface for density optimization, where we can pass in a molecule as `.xyz` file and a path to a trained model checkpoint (as well as a config used in `get_sample_from_mol` to correctly create the `OFData` object). The `.xyz` file is read by `pyscf` to create a molecule object. Afterwards, it's only a matter of calling our `_denop` function and printing the results.

You should obtain the exact same energy results as in the [Density Optimization notebook](tutorial_6_density_optimization.ipynb).

In [ ]:
@set_default_torch_dtype(torch.float64)
def denop(xyz_input_file: str, model_config_path: str, model_path: str):
    """
    Performs density optimization on the given molecule file (in .xyz format)
    using a pre-trained MLDFT model.
    """
    config = OmegaConf.load(model_config_path)
    assert isinstance(config, DictConfig), "Expected DictConfig"

    # no need to waste GPU resources here, CPU is fine
    model = MLDFTLitModuleSimplified.load_from_checkpoint(model_path, map_location="cpu")
    model.eval()
    model.to(torch.float64)

    mole = gto.M(atom=str(xyz_input_file))
    sample = get_sample_from_mol(mole, config)

    print("🎈 Running density optimization")
    energies, coeffs, converged = _denop(model, sample, sample.mol)

    if converged:
        print("✅ Density optimization converged")
        print("Final energies:", energies)
        # print("Final coeffs:", coeffs)
    else:
        print("⚠️ Density optimization did not converge")


denop(MOLECULE_FILE, MODEL_CONFIG_PATH, MODEL_PATH)